# CargoSense — Phase 8: Feature Engineering & Baseline Modeling Preparation
**Pipeline Development, Leakage Prevention, Chronological Splitting, and Preprocessing Architecture**  
*Curriculum Focus: Feature Engineering, Data Preprocessing (ColumnTransformer, StandardScaler, OneHotEncoder), Competing Candidate Group Splitting, and Class Imbalance Formulation*

---

## 1. Executive Context & Objective
In **Phase 6**, we generated 2,181 candidate consolidation opportunities across 727 shipments and derived the `Became_FTL` ground-truth target strictly from subsequent operational loading and dispatch events.

In **Phase 7**, we conducted statistical analysis, evaluated truck density envelopes, and established a 14-feature shortlist available at the 18:00 planning cutoff.

In **Phase 8**, our goals are:
1. Validate the 14-feature shortlist against `ode_opportunities.csv` and verify pre-cutoff availability.
2. Prevent data leakage: strictly exclude `candidate_strategy`, post-cutoff fields, and memorizing IDs.
3. Engineer candidate physical interaction features (density, fill ratios).
4. Construct a **chronological shipment-grouped train/val/test split** keeping competing candidates together.
5. Build and fit a scikit-learn `ColumnTransformer` preprocessing pipeline exclusively on training data.
6. Export model-ready dataset partitions and serialization artifacts for Phase 9 baseline modeling.

In [1]:
import os
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

print("Phase 8 Environment Initialized Successfully.")

Phase 8 Environment Initialized Successfully.


## 2. Load ODE Dataset & Schema Verification
We load `ode_opportunities.csv` and inspect dataset dimensions, column types, and null counts.

In [2]:
data_dir = Path('../data') if Path('../data').exists() else Path('data')
ode = pd.read_csv(data_dir / 'processed' / 'ode_opportunities.csv')

print(f"ODE Opportunities Shape: {ode.shape[0]:,} rows x {ode.shape[1]} columns")
print(f"Unique Shipments: {ode['shipment_id'].nunique():,}")
print(f"Candidates per Shipment: {ode['shipment_id'].value_counts().unique()}")
print(f"Target Became_FTL Prevalence: {ode['Became_FTL'].mean():.4f} ({ode['Became_FTL'].sum()} positives)")
ode.head(3)

ODE Opportunities Shape: 2,181 rows x 39 columns
Unique Shipments: 727
Candidates per Shipment: [3]
Target Became_FTL Prevalence: 0.0853 (186 positives)


,opportunity_id,evaluation_timestamp,planning_run_id,shipment_id,truck_id,route_id,truck_type,max_payload_kg,max_volume_m3,planned_weight_kg,...,candidate_added_volume_m3,projected_weight_kg,projected_volume_m3,projected_weight_util_pct,projected_volume_util_pct,is_feasible,mean_demand_last_7d_route,day_of_week,days_into_simulation,Became_FTL
0,OPP-SHP-20240101-001-RUN-20240101-balanced,2024-01-01 23:00:00,RUN-20240101,SHP-20240101-001,TRK-001,RT-MUM-PUN,14ft LCV,3500.0,18.0,17.8,...,17.868678,3499.986,17.924328,99.9996,99.5796,1,7.1429,0,1,0
1,OPP-SHP-20240101-001-RUN-20240101-volume_optim...,2024-01-01 23:00:00,RUN-20240101,SHP-20240101-001,TRK-001,RT-MUM-PUN,14ft LCV,3500.0,18.0,17.8,...,17.944146,1607.722,17.999796,45.9349,99.9989,1,7.1429,0,1,0
2,OPP-SHP-20240101-001-RUN-20240101-weight_optim...,2024-01-01 23:00:00,RUN-20240101,SHP-20240101-001,TRK-001,RT-MUM-PUN,14ft LCV,3500.0,18.0,17.8,...,9.841902,3499.975,9.897552,99.9993,54.9864,1,7.1429,0,1,0


## 3. Audit Baseline Features & Leakage Exclusion
Verify that all 14 shortlisted features exist and have zero missing values. Confirm the exclusion of `candidate_strategy` to prevent trivial target leakage.

In [3]:
baseline_numeric = [
    "remaining_capacity_kg", "remaining_capacity_m3",
    "candidate_added_weight_kg", "candidate_added_volume_m3",
    "projected_weight_util_pct", "projected_volume_util_pct",
    "n_candidate_skus", "candidate_total_qty",
    "mean_demand_last_7d_route", "route_transit_hours",
    "days_into_simulation"
]

baseline_categorical = ["truck_type", "route_id", "day_of_week"]
shared_baseline_features = baseline_numeric + baseline_categorical

# Check nulls
null_summary = ode[shared_baseline_features].isnull().sum()
print("Null count across baseline features:")
print(null_summary)

# Strategy vs Target Leakage Demonstration
print("\nDemonstrating why candidate_strategy MUST be excluded:")
display(pd.crosstab(ode['candidate_strategy'], ode['Became_FTL'], normalize='index'))

Null count across baseline features:
remaining_capacity_kg        0
remaining_capacity_m3        0
candidate_added_weight_kg    0
candidate_added_volume_m3    0
projected_weight_util_pct    0
projected_volume_util_pct    0
n_candidate_skus             0
candidate_total_qty          0
mean_demand_last_7d_route    0
route_transit_hours          0
days_into_simulation         0
truck_type                   0
route_id                     0
day_of_week                  0
dtype: int64

Demonstrating why candidate_strategy MUST be excluded:


Became_FTL,0,1
candidate_strategy,,
balanced,0.744154,0.255846
volume_optimized,1.000000,0.000000
weight_optimized,1.000000,0.000000


## 4. Engineer Custom Interaction Features
Derive physical ratio and density metrics available for Phase 9 custom model runs:
- `cargo_density_kg_m3`: Proposed cargo mass divided by proposed pack volume.
- `weight_fill_ratio`: Proposed mass divided by remaining payload capacity.
- `volume_fill_ratio`: Proposed volume divided by remaining volume capacity.
- `avg_unit_weight_kg`: Proposed mass divided by piece quantity.

In [4]:
df_eng = ode.copy()
df_eng["cargo_density_kg_m3"] = df_eng["candidate_added_weight_kg"] / df_eng["candidate_added_volume_m3"].replace(0, np.nan)
df_eng["weight_fill_ratio"] = df_eng["candidate_added_weight_kg"] / df_eng["remaining_capacity_kg"].replace(0, np.nan)
df_eng["volume_fill_ratio"] = df_eng["candidate_added_volume_m3"] / df_eng["remaining_capacity_m3"].replace(0, np.nan)
df_eng["avg_unit_weight_kg"] = df_eng["candidate_added_weight_kg"] / df_eng["candidate_total_qty"].replace(0, np.nan)

custom_cols = ["cargo_density_kg_m3", "weight_fill_ratio", "volume_fill_ratio", "avg_unit_weight_kg"]
display(df_eng[custom_cols].describe().T[['mean', 'std', 'min', 'max']])

,mean,std,min,max
cargo_density_kg_m3,217.383401,105.357940,85.326276,359.658943
weight_fill_ratio,0.813343,0.263594,0.415810,1.000000
volume_fill_ratio,0.865848,0.188881,0.533885,1.000000
avg_unit_weight_kg,0.453984,0.282948,0.150839,0.856861


## 5. Leakage-Safe Chronological Shipment-Grouped Splitting
Split the 89 planning runs chronologically: 70% Train, 15% Validation, 15% Test.
Ensure all 3 competing candidates for each shipment remain strictly clustered within the same partition.

In [5]:
unique_runs = sorted(df_eng["planning_run_id"].unique())
n_runs = len(unique_runs)

n_train_runs = int(n_runs * 0.70)
n_val_runs = int(n_runs * 0.15)
train_runs = unique_runs[:n_train_runs]
val_runs = unique_runs[n_train_runs:n_train_runs + n_val_runs]
test_runs = unique_runs[n_train_runs + n_val_runs:]

train_df = df_eng[df_eng["planning_run_id"].isin(train_runs)].copy()
val_df = df_eng[df_eng["planning_run_id"].isin(val_runs)].copy()
test_df = df_eng[df_eng["planning_run_id"].isin(test_runs)].copy()

print(f"Train: {len(train_runs)} runs, {train_df['shipment_id'].nunique()} shipments, {len(train_df)} rows, pos={train_df['Became_FTL'].sum()} ({train_df['Became_FTL'].mean()*100:.2f}%)")
print(f"Val:   {len(val_runs)} runs, {val_df['shipment_id'].nunique()} shipments, {len(val_df)} rows, pos={val_df['Became_FTL'].sum()} ({val_df['Became_FTL'].mean()*100:.2f}%)")
print(f"Test:  {len(test_runs)} runs, {test_df['shipment_id'].nunique()} shipments, {len(test_df)} rows, pos={test_df['Became_FTL'].sum()} ({test_df['Became_FTL'].mean()*100:.2f}%)")

# Overlap Verification
assert len(set(train_df['shipment_id']).intersection(set(val_df['shipment_id']))) == 0
assert len(set(train_df['shipment_id']).intersection(set(test_df['shipment_id']))) == 0
assert len(set(val_df['shipment_id']).intersection(set(test_df['shipment_id']))) == 0
print("Zero shipment overlap confirmed across all partitions.")

Train: 62 runs, 511 shipments, 1533 rows, pos=141 (9.20%)
Val:   13 runs, 107 shipments, 321 rows, pos=23 (7.17%)
Test:  14 runs, 109 shipments, 327 rows, pos=22 (6.73%)
Zero shipment overlap confirmed across all partitions.


## 6. Preprocessing Pipeline Definition & Fitting
Build a scikit-learn `ColumnTransformer` with `SimpleImputer` and `StandardScaler` for numeric features, and `SimpleImputer` and `OneHotEncoder` for categorical features. Fit strictly on `train_df`.

In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, baseline_numeric),
        ("cat", categorical_transformer, baseline_categorical)
    ],
    remainder="drop"
)

# Fit exclusively on training data
preprocessor.fit(train_df[shared_baseline_features])

# Transform matrices
X_train = preprocessor.transform(train_df[shared_baseline_features])
X_val   = preprocessor.transform(val_df[shared_baseline_features])
X_test  = preprocessor.transform(test_df[shared_baseline_features])

y_train = train_df['Became_FTL'].values
y_val   = val_df['Became_FTL'].values
y_test  = test_df['Became_FTL'].values

print(f"Transformed X_train shape: {X_train.shape}")
print(f"Transformed X_val shape:   {X_val.shape}")
print(f"Transformed X_test shape:  {X_test.shape}")

Transformed X_train shape: (1533, 31)
Transformed X_val shape:   (321, 31)
Transformed X_test shape:  (327, 31)


## 7. Class Imbalance Hyperparameter Formulation
With an 8.53% overall positive rate (~9.87:1 imbalance ratio), calculate standard cost-sensitive hyperparameters for Phase 9:
- **Random Forest**: `class_weight='balanced'`
- **XGBoost**: `scale_pos_weight = (n_neg / n_pos)`

In [7]:
n_pos_train = int(np.sum(y_train == 1))
n_neg_train = int(np.sum(y_train == 0))
scale_pos_weight = n_neg_train / n_pos_train

print(f"Training Class Counts: Positive = {n_pos_train}, Negative = {n_neg_train}")
print(f"Calculated scale_pos_weight for XGBoost: {scale_pos_weight:.4f}")
print(f"Recommended Random Forest class_weight: 'balanced'")

Training Class Counts: Positive = 141, Negative = 1392
Calculated scale_pos_weight for XGBoost: 9.8723
Recommended Random Forest class_weight: 'balanced'


## 8. Export Summary & Phase 9 Handoff
All artifacts, partitions, serialized pipelines, and documentation are saved and ready for Phase 9 model benchmarking:
1. `data/processed/phase8_train.csv`
2. `data/processed/phase8_val.csv`
3. `data/processed/phase8_test.csv`
4. `models/phase8_preprocessor.joblib`
5. `reports/phase8_feature_engineering.md`